### Import the necessary database

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [3]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [4]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [5]:
ds

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164, lat: 90, lon: 180)>
 dask.array<open_dataset-df1d26385c17deb67f2ced416cec5bf3trend, shape=(164, 90, 180), dtype=float64, chunksize=(164, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163, lat: 90, lon: 180)>
 dask.array<open_dataset-a13757215e76170d6f332760606c3821trend, shape=(163, 90, 180), dtype=float64, chunksize=(163, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162, lat: 90, lon: 180)>
 dask.array<open_d

In [6]:
# flip the longitude to -180 to 180
ds_adj = {}
for key in ds.keys():
    ds_adj[key] = preprosess.adjust_longitude(ds[key],ds[key].lon)

In [7]:
ds_adj

{'ICV_trend_10yr': (array([[[-0.00876186, -0.0087619 , -0.00877973, ..., -0.00876184,
           -0.00876181, -0.00876182],
          [-0.00841684, -0.00842867, -0.00844424, ..., -0.00839057,
           -0.00839713, -0.00840502],
          [-0.00705928, -0.00710348, -0.00715207, ..., -0.00694285,
           -0.00697782, -0.00701509],
          ...,
          [-0.05359884, -0.05381129, -0.05397888, ..., -0.05292523,
           -0.05314104, -0.05338639],
          [-0.05438904, -0.0545001 , -0.05462539, ..., -0.05402935,
           -0.0541402 , -0.05427799],
          [-0.0546163 , -0.05461719, -0.05480954, ..., -0.05440526,
           -0.05440612, -0.05461542]],
  
         [[-0.01181036, -0.01181041, -0.01183444, ..., -0.01181033,
           -0.01181029, -0.01181031],
          [-0.0113453 , -0.01136124, -0.01138223, ..., -0.01130988,
           -0.01131873, -0.01132936],
          [-0.00951541, -0.00957498, -0.00964048, ..., -0.00935847,
           -0.0094056 , -0.00945584],
         

In [8]:
ds_temp = {}
for key in ds.keys():
    ds_temp[key] = xr.DataArray(
        ds_adj[key][0],
        coords={'segment': ds[key].segment, 'lat': ds[key].lat, 'lon': ds_adj[key][1]},
        dims=('segment', 'lat', 'lon')
    )

In [9]:
ds_temp

{'ICV_trend_10yr': <xarray.DataArray (segment: 164, lat: 90, lon: 180)>
 array([[[-0.00876186, -0.0087619 , -0.00877973, ..., -0.00876184,
          -0.00876181, -0.00876182],
         [-0.00841684, -0.00842867, -0.00844424, ..., -0.00839057,
          -0.00839713, -0.00840502],
         [-0.00705928, -0.00710348, -0.00715207, ..., -0.00694285,
          -0.00697782, -0.00701509],
         ...,
         [-0.05359884, -0.05381129, -0.05397888, ..., -0.05292523,
          -0.05314104, -0.05338639],
         [-0.05438904, -0.0545001 , -0.05462539, ..., -0.05402935,
          -0.0541402 , -0.05427799],
         [-0.0546163 , -0.05461719, -0.05480954, ..., -0.05440526,
          -0.05440612, -0.05461542]],
 
        [[-0.01181036, -0.01181041, -0.01183444, ..., -0.01181033,
          -0.01181029, -0.01181031],
         [-0.0113453 , -0.01136124, -0.01138223, ..., -0.01130988,
          -0.01131873, -0.01132936],
         [-0.00951541, -0.00957498, -0.00964048, ..., -0.00935847,
          -0

In [10]:
lat = ds_temp["ICV_trend_10yr"].lat
lon = ds_temp["ICV_trend_10yr"].lon
# Extratropical South Pacific region
# SO region 
lat1 = -65
lat2 = -50
lon1 = -180
lon2 = 180
# select the region

ds_SO_masked = {}
for key in ds.keys():
    ds_SO_masked[key] = data_process.selreg(ds_temp[key],lat, lon, lat1, lat2, lon1, lon2)

In [11]:
ds_SO_masked

{'ICV_trend_10yr': (<xarray.DataArray (segment: 164, lat: 8, lon: 180)>
  array([[[-1.26639470e-03, -1.01341319e-03, -3.42919216e-03, ...,
           -2.69931944e-03, -2.38094458e-03, -1.51937620e-03],
          [ 1.38296411e-03,  1.59524879e-03, -1.41207454e-03, ...,
           -1.54477588e-04,  2.91632206e-04,  1.17067943e-03],
          [ 4.95935971e-03,  5.04532482e-03,  4.05438220e-03, ...,
            2.70814685e-03,  3.18872553e-03,  4.87339460e-03],
          ...,
          [ 1.06508766e-02,  9.97105318e-03,  8.16591184e-03, ...,
            7.13827011e-03,  7.41019512e-03,  1.13307000e-02],
          [ 9.64437840e-03,  8.44482774e-03,  6.17528428e-03, ...,
            7.60644402e-03,  7.60765992e-03,  1.08439291e-02],
          [-6.97282515e-01, -7.25217315e-01, -7.22545065e-01, ...,
           -5.94681325e-01, -6.15461565e-01, -6.69347715e-01]],
  
         [[-1.70700907e-03, -1.36600818e-03, -4.62230466e-03, ...,
           -3.63848867e-03, -3.20934223e-03, -2.04800996e-03],

In [12]:
# calculate the regional mean
ds_SO_mean = {}
for key in ds_SO_masked.keys():
    ds_SO_mean[key] = data_process.calc_weighted_mean(ds_SO_masked[key][0])

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [13]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [14]:
# calculate the regional mean's percentile
# 5%---[0]
unforced_trend_SO_lower_percentile = {}

# 95%---[1]
unforced_trend_SO_upper_percentile = {}

for key in ds_SO_masked.keys():
    unforced_trend_SO_lower_percentile[key], unforced_trend_SO_upper_percentile[key] = calc_percentile(ds_SO_mean[key], 5)
    

In [15]:
unforced_trend_SO_lower_percentile

{'ICV_trend_10yr': -0.24616394742627465,
 'ICV_trend_11yr': -0.2254561296731142,
 'ICV_trend_12yr': -0.21399684314163614,
 'ICV_trend_13yr': -0.20368432678913354,
 'ICV_trend_14yr': -0.19148392840370299,
 'ICV_trend_15yr': -0.18018207401953099,
 'ICV_trend_16yr': -0.17750807750647468,
 'ICV_trend_17yr': -0.16069008107820953,
 'ICV_trend_18yr': -0.15042959631522745,
 'ICV_trend_19yr': -0.1530952365372739,
 'ICV_trend_20yr': -0.14601307484230203,
 'ICV_trend_21yr': -0.13896557959859185,
 'ICV_trend_22yr': -0.13621177814927773,
 'ICV_trend_23yr': -0.13131366564762564,
 'ICV_trend_24yr': -0.1292340175759312,
 'ICV_trend_25yr': -0.1275284755186547,
 'ICV_trend_26yr': -0.12093469636339235,
 'ICV_trend_27yr': -0.12151998901124998,
 'ICV_trend_28yr': -0.12810706099625702,
 'ICV_trend_29yr': -0.1243896635433168,
 'ICV_trend_30yr': -0.12221676101704144,
 'ICV_trend_31yr': -0.12447363405620675,
 'ICV_trend_32yr': -0.12422341396128551,
 'ICV_trend_33yr': -0.11880628386244237,
 'ICV_trend_34yr': -0

In [16]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_SO_lower_percentile.keys())
unforced_trend_SO_lower_percentile_da = xr.DataArray(
	list(unforced_trend_SO_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_SO_upper_percentile_da = xr.DataArray(
	list(unforced_trend_SO_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [17]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_SO_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_SO_trend_lower_percentile.nc')
unforced_trend_SO_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_SO_trend_upper_percentile.nc')
